Published RNA studies used:
1) GTEx V8 counts per tissue - Processed to take the mean CPM per gene per (combined) tissue
2) Loy et al. (2024), Children inflammatory syndromes - All processed sequencing data
3) Moufarrej et al. (2022), Pre-eclampsia - All processed sequencing data from all cohorts with available PreQC counts data
4) Chalassani et al. (2021), Liver diseases and healthy controls - All processed sequencing data from all cohorts
5) Toden et al. (2020), Alzheimer's disease and healthy controls - All processed sequencing data from all cohorts


Half-life study:
Agarwal, V., Kelley, D.R. The genetic and biochemical determinants of mRNA degradation rates in mammals. Genome Biol 23, 245 (2022). https://doi.org/10.1186/s13059-022-02811-x
using the "13059_2022_2811_MOESM3_ESM_Humans_Average_AllNames.csv" file generated from 06_Degradation-vs-Matrix_Degradation-top-genes.ipynb

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import matplotlib as mpl
from matplotlib import rcParams
from matplotlib.patches import Patch

mpl.rcParams['svg.fonttype'] = 'none'         # keep text as text (editable)
mpl.rcParams['pdf.fonttype'] = 42             # optional: good if you also export PDF
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

rcParams.update({
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.title_fontsize": 11,
    "legend.fontsize": 10
})
sns.set(style="whitegrid")

#############################################
# 1. Load half-life reference
#############################################

half_life_file = "13059_2022_2811_MOESM3_ESM_Humans_Average_AllNames.csv"

half_life = pd.read_csv(half_life_file)
half_life = half_life.rename(columns={
    "original_gene_name": "Gene_name_orig",
    "ensembl_id": "Ensembl_Gene_Id",
    "recovered_gene_name": "Gene_name_rec",
    "half-life (PC1)": "Average_Half_Life"
})

half_life["Ensembl_Gene_Id"] = half_life["Ensembl_Gene_Id"].astype(str).str.split(".").str[0]

#############################################
# 2. Half-life binning
#############################################

def categorize_half_life_sd(df):
    mean_hl = df["Average_Half_Life"].mean()
    std_hl  = df["Average_Half_Life"].std()
    return df["Average_Half_Life"].apply(lambda x:
        "Very Low"  if x <= mean_hl - std_hl else
        "Low"       if x <= mean_hl - 0.5 * std_hl else
        "Medium"    if x <  mean_hl + 0.5 * std_hl else
        "High"      if x <  mean_hl + std_hl else
        "Very High"
    )

cat_order = ["Very Low", "Low", "Medium", "High", "Very High"]


#############################################
# 3. GTEx processing
#############################################

def process_gtex(file_path):
    df = pd.read_csv(file_path, sep="\t")

    df["Ensembl_Gene_Id"] = df["EnsemblID"].astype(str).str.split(".").str[0]

    long_df = df.melt(
        id_vars=["GeneID", "EnsemblID", "Ensembl_Gene_Id"],
        var_name="tissue",
        value_name="mean_expression"
    )

    long_df = long_df[long_df["mean_expression"] > 0].copy()

    merged = pd.merge(long_df, half_life, on="Ensembl_Gene_Id", how="inner")
    if merged.empty:
        print("No overlap between GTEx and half-life reference!")
        return None

    merged["half_life_bin"] = categorize_half_life_sd(merged)
    merged["half_life_bin"] = pd.Categorical(
        merged["half_life_bin"], categories=cat_order, ordered=True
    )

    merged["log2_mean_expression"] = np.log2(merged["mean_expression"] + 1)
    merged["dataset"] = "GTEx"

    return merged[["dataset", "half_life_bin", "log2_mean_expression"]]


#############################################
# 4. Process the other datasets (from script 1)
#############################################

def process_counts(file_path, dataset_name, id_type="auto"):
    df = pd.read_csv(file_path, sep="\t|,", engine="python")

    first_col = df.columns[0]
    if "ensg" in df[first_col].astype(str).str.lower().iloc[0]:
        id_type = "ensembl"
    else:
        id_type = "gene"

    count_matrix = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").fillna(0)
    library_sizes = count_matrix.sum(axis=0)
    cpm = count_matrix.div(library_sizes) * 1e6
    df["mean_expression"] = cpm.mean(axis=1).values

    if id_type == "ensembl":
        df["Ensembl_Gene_Id"] = df[first_col].astype(str).str.split(".").str[0]
        merged = pd.merge(df, half_life, on="Ensembl_Gene_Id", how="inner")
    else:
        df = df.rename(columns={first_col: "Gene_name"})
        merged1 = pd.merge(df, half_life, left_on="Gene_name", right_on="Gene_name_orig", how="inner")
        merged2 = pd.merge(df, half_life, left_on="Gene_name", right_on="Gene_name_rec", how="inner")
        merged = pd.concat([merged1, merged2], ignore_index=True).drop_duplicates(subset=["Gene_name"])

    if merged.empty:
        print(f"No overlap for {dataset_name} ({file_path})")
        return None

    merged["half_life_bin"] = categorize_half_life_sd(merged)
    merged["half_life_bin"] = pd.Categorical(merged["half_life_bin"], categories=cat_order, ordered=True)
    merged = merged[merged["mean_expression"] > 0].copy()
    merged["log2_mean_expression"] = np.log2(merged["mean_expression"] + 1)
    merged["dataset"] = dataset_name

    return merged[["dataset", "half_life_bin", "log2_mean_expression"]]


def process_counts_multi(file_paths, dataset_name, id_type="auto"):
    """
    Accepts a single file path or a list of file paths.
    Pools all files into one dataset label (dataset_name).
    """
    if isinstance(file_paths, (str, os.PathLike)):
        file_paths = [file_paths]

    dfs = []
    for fp in file_paths:
        out = process_counts(fp, dataset_name, id_type=id_type)
        if out is not None and not out.empty:
            dfs.append(out)

    if not dfs:
        print(f"No usable files for {dataset_name}")
        return None

    return pd.concat(dfs, ignore_index=True)


datasets = {
    "Loy": "GSE255555_pedInflam_counts.csv",
    "Moufarrej": "Moufarrej_merged_Stanford_Validation2.csv",
    "Toden": [
        "Toden_AD/All-AD-Aligned/All-AD_Counts_v46_Clean.txt",
        "Toden_AD/All-NCI-Aligned/All-NCI_Counts_v46_Clean.txt",
    ],
    "Chalasani": [
        "Chalasani_Liver/All-Healthy-Aligned/All-Healthy_Counts_v46_Clean.txt",
        "Chalasani_Liver/All-NASH-Aligned/All-NASH_Counts_v46_Clean.txt",
        "Chalasani_Liver/All-NAFLD-Aligned/All-NAFLD_Counts_v46_Clean.txt",
    ],
}


#############################################
# 5. Run all processing
#############################################

all_dfs = []

# GTEx first
gtex_df = process_gtex("20251118_GTEx_Both_MeanCPM_Tissues_Both.tsv")
if gtex_df is not None:
    all_dfs.append(gtex_df)

# Other dataset groups (now supports single path OR list of paths)
for name, paths in datasets.items():
    df = process_counts_multi(paths, name)
    if df is not None:
        all_dfs.append(df)

final_df = pd.concat(all_dfs, ignore_index=True)


#############################################
# 6. Make combined violin plot
#############################################

plt.figure(figsize=(10, 5))
ax = sns.violinplot(
    data=final_df,
    x="dataset",
    y="log2_mean_expression",
    hue="half_life_bin",
    hue_order=cat_order,
    cut=0,
    inner="box",
    palette="colorblind",
    width=0.8,
    dodge=True
)

group_means = final_df.groupby(["dataset", "half_life_bin"])["log2_mean_expression"].mean()

# --- Overlay group means as red dots (correct placement)
dataset_order = list(pd.unique(final_df["dataset"]))
n_hue = len(cat_order)
violin_width = 0.8
step = violin_width / n_hue

for i, ds in enumerate(dataset_order):
    for j, hb in enumerate(cat_order):
        if (ds, hb) not in group_means.index:
            continue
        x = i + (j - (n_hue - 1) / 2) * step
        y = group_means.loc[(ds, hb)]
        ax.scatter(x, y, s=35, color="red", edgecolor="black", zorder=10)

plt.xlabel("Dataset")
plt.ylabel("Log2 Mean Expression (log2[CPM + 1])")
ax.yaxis.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)
ax.set_axisbelow(True)
ax.tick_params(axis="x")
ax.tick_params(axis="y")

# --- Clean custom legend (avoids duplicated/multiline entries)
palette = sns.color_palette("colorblind", n_colors=len(cat_order))
legend_handles = [Patch(facecolor=palette[k], label=cat_order[k]) for k in range(len(cat_order))]
legend_handles.append(
    Line2D([0], [0], marker="o", color="red", label="Mean",
           markerfacecolor="red", markeredgecolor="black",
           markersize=6, linestyle="")
)

ax.legend(legend_handles, [h.get_label() for h in legend_handles],
          title="Half-life bin",
          bbox_to_anchor=(1.0, 1), loc="upper left", borderaxespad=0.4)

plt.tight_layout()
fig = plt.gcf()  # get current figure
svg_path = "GTEx_and_AllDatasets_HalfLife_Violin.svg"
fig.savefig(svg_path, format="svg", bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_1623353/3578317891.py:207: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_means = final_df.groupby(["dataset", "half_life_bin"])["log2_mean_expression"].mean()


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata, norm

def jt_fast_rank(
    sub_df,
    value_col="log2_mean_expression",
    group_col="half_life_bin",
    order=None,
    alternative="increasing",
):
    """
    Fast Jonckheere–Terpstra trend test using ranks (O(N log N)).

    - Works with unequal group sizes.
    - Uses midranks to handle ties reasonably.
    - Normal approximation for z and p-value.

    Returns:
      J, z, p_one_sided, used_bins, ns
    """
    if order is None:
        order = sorted(sub_df[group_col].dropna().unique().tolist())

    # Keep only needed columns and drop missing
    df = sub_df[[group_col, value_col]].dropna().copy()

    # Enforce ordering and drop empty groups
    df[group_col] = pd.Categorical(df[group_col], categories=order, ordered=True)
    df = df[df[group_col].notna()]

    # If fewer than 2 groups present, bail
    present = [g for g in order if (df[group_col] == g).any()]
    if len(present) < 2:
        return np.nan, np.nan, np.nan, present, [int((df[group_col] == g).sum()) for g in present]

    # Map ordered groups to integer codes 0..k-1 (in the specified order)
    df = df.sort_values(group_col)
    group_codes = df[group_col].cat.codes.to_numpy()
    x = df[value_col].to_numpy()

    # Midranks across pooled data (handles ties)
    r = rankdata(x, method="average")

    # Group sizes and rank sums per group in order
    k = len(order)
    n = np.bincount(group_codes, minlength=k).astype(float)
    R = np.bincount(group_codes, weights=r, minlength=k).astype(float)

    # Restrict to groups that actually exist (non-zero n)
    mask = n > 0
    n = n[mask]
    R = R[mask]
    used_bins = [g for g in order if (df[group_col] == g).any()]
    k_eff = len(n)
    if k_eff < 2:
        return np.nan, np.nan, np.nan, used_bins, n.astype(int).tolist()

    # Compute J from rank sums:
    # J = sum_{j} [ R_j * sum_{i<j} n_i ] - sum_{j} [ n_j * (n_j+1)/2 * sum_{i<j} n_i ]
    # This works with midranks and avoids O(N^2)
    cum_n_before = np.cumsum(n) - n  # sum_{i<j} n_i for each j
    J = np.sum(R * cum_n_before) - np.sum((n * (n + 1) / 2.0) * cum_n_before)

    # Mean and variance under H0 (no ties variance; ranks already midranked)
    N = np.sum(n)
    mean_J = (N**2 - np.sum(n**2)) / 4.0

    s1 = np.sum(n * (n - 1) * (2 * n + 5))
    var_J = (N * (N - 1) * (2 * N + 5) - s1) / 72.0
    if var_J <= 0:
        return J, np.nan, np.nan, used_bins, n.astype(int).tolist()

    z = (J - mean_J) / np.sqrt(var_J)

    # p-values
    if alternative == "increasing":
        p = 1.0 - norm.cdf(z)
    elif alternative == "decreasing":
        p = norm.cdf(z)
    else:
        p = 2.0 * min(norm.cdf(z), 1.0 - norm.cdf(z))

    return float(J), float(z), float(p), used_bins, n.astype(int).tolist()


# ---- Run JT per dataset and print
jt_rows = []
for ds, sub in final_df.dropna(subset=["dataset", "half_life_bin", "log2_mean_expression"]).groupby("dataset"):
    J, z, p, used_bins, ns = jt_fast_rank(
        sub,
        value_col="log2_mean_expression",
        group_col="half_life_bin",
        order=cat_order,
        alternative="increasing"
    )
    jt_rows.append({
        "dataset": ds,
        "JT_stat": J,
        "z": z,
        "p_trend_one_sided": p,
        "n_per_bin": ",".join(map(str, ns))
    })

jt_df = pd.DataFrame(jt_rows).sort_values("p_trend_one_sided", na_position="last")

def fmt_p(p):
    if pd.isna(p): return "NA"
    if p == 0 or p < 1e-300: return "<1e-300"
    return f"{p:.2e}"

print("\n=== Fast Jonckheere–Terpstra trend test (one-sided: increasing) ===")
print(jt_df.assign(p_trend_one_sided=jt_df["p_trend_one_sided"].map(fmt_p))
          [["dataset","JT_stat","z","p_trend_one_sided","n_per_bin"]]
          .to_string(index=False))



=== Fast Jonckheere–Terpstra trend test (one-sided: increasing) ===
  dataset      JT_stat            z p_trend_one_sided                      n_per_bin
Chalasani 1.131142e+13 8.418271e+06           <1e-300      6500,5919,16160,6073,6601
     GTEx 1.202652e+16 2.797838e+08           <1e-300 65575,60056,163016,61018,66089
      Loy 3.778981e+11 1.517671e+06           <1e-300       2119,1926,5260,1974,2130
Moufarrej 4.003471e+11 1.574673e+06           <1e-300       2149,1954,5344,1993,2158
    Toden 3.374631e+12 4.557949e+06           <1e-300      4379,3984,10882,4068,4416
